# **REFERENCE MATCHING PIPELINE**

---

## Mục tiêu
Pipeline này thực hiện matching giữa các BibTeX entries (từ `refs.bib`) với các arXiv entries (từ `references.json`) để xây dựng đồ thị trích dẫn giữa các bài báo.

## Bài toán
- **Loại bài toán**: Ranking/Retrieval
- **Input**: Một BibTeX entry (key, title, authors, year, venue)
- **Output**: Danh sách xếp hạng top 5 arXiv IDs
- **Metric**: Mean Reciprocal Rank (MRR@5)

## Data Configuration (từ label.json)
- **Manual Subset**: 5 publications đã gắn nhãn thủ công (trong `processed/`)
- **Auto Subset**: 500 publications cần gắn nhãn tự động (trong `data/`)

## Pipeline Steps
1. **Data Loading & Cleaning**: Load dữ liệu từ cả 2 folders, chuẩn hóa text
2. **Exploratory Data Analysis (EDA)**: Phân tích dữ liệu để hiểu đặc trưng
3. **Data Labeling**: Manual (5 pubs, ≥20 pairs/pub) + Auto (10% còn lại)
4. **Feature Engineering**: Tạo features từ title, author, year similarity
5. **Data Split**: test=(1 manual + 1 auto), valid=(1 manual + 1 auto), train=còn lại
6. **Model Training**: Huấn luyện Gradient Boosting Classifier
7. **Evaluation**: Đánh giá MRR@5 trên test set
8. **Prediction**: Xuất file pred.json cho mỗi publication

In [1]:
# Import các thư viện cần thiết
import sys
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import importlib
import matching
importlib.reload(matching)
from matching import (
    DataLoader, FeatureExtractor, DataLabeler, 
    ReferenceMatcherModel, Evaluator, ReferenceMatchingPipeline,
    TextCleaner, BibEntry, ArxivEntry, EDAUtils
)

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = BASE_DIR / "processed"
LABEL_JSON = PROCESSED_DIR / "label.json"
print(f"Base directory: {BASE_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Label config: {LABEL_JSON}")

# Load label configuration
with open(LABEL_JSON, 'r', encoding='utf-8') as f:
    label_config = json.load(f)

print(f"\n   Label Configuration")
print(f"Manual subset: {label_config['manual_subset']['count']} publications")
print(f"  Papers: {label_config['manual_subset']['papers']}")
print(f"Auto subset: {label_config['auto_subset']['count']} publications")

Base directory: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195
Processed directory: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/processed
Label config: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/processed/label.json

   Label Configuration
Manual subset: 5 publications
  Papers: ['2412-18790', '2412-19279', '2412-19837', '2412-20155', '2412-20242']
Auto subset: 500 publications


## 1. Data Loading & Exploration

### 1.1 Kiểm tra dữ liệu Manual Labels

Bước đầu tiên trong quy trình chuẩn bị dữ liệu là xác minh tính toàn vẹn của tập dữ liệu nhãn thủ công, đóng vai trò quyết định trong việc đánh giá hiệu năng thực tế (Model Evaluation) của hệ thống khớp nối.

Đoạn mã dưới đây thực hiện tải và thống kê số lượng cặp khớp nối (positive matches) từ 5 bài báo được chọn lọc kỹ lưỡng. Mục tiêu là đảm bảo mỗi bài báo có đủ số lượng mẫu (> 20 cặp) để đại diện cho các trường hợp trích dẫn đa dạng.

In [2]:
# Load and display manual labels for 5 publications.
manual_labels_all = {}

for pub_id in label_config['manual_subset']['papers']:
    labels_path = PROCESSED_DIR / pub_id / 'labels_manual.json'
    if labels_path.exists():
        with open(labels_path, 'r', encoding='utf-8') as f:
            labels = json.load(f)
            manual_labels_all[pub_id] = labels
            print(f"\n=== {pub_id} ===")
            print(f"Number of labeled pairs: {len(labels)}")
            # Show first 5 pairs
            for i, (bib_key, arxiv_id) in enumerate(list(labels.items())[:5]):
                print(f"  {bib_key} -> {arxiv_id}")
            if len(labels) > 5:
                print(f"  ... and {len(labels) - 5} more pairs")
    else:
        print(f"[WARNING] labels_manual.json not found for {pub_id}")

# Statistics
total_manual_pairs = sum(len(labels) for labels in manual_labels_all.values())
print(f"\n=== Manual Labels Summary ===")
print(f"Total publications with manual labels: {len(manual_labels_all)}")
print(f"Total labeled pairs: {total_manual_pairs}")
print(f"Average pairs per publication: {total_manual_pairs/len(manual_labels_all):.1f}")


=== 2412-18790 ===
Number of labeled pairs: 32
  lyle2024normalization -> 2407-01800
  lewandowski2024learning -> 2406-06811
  elsayed2024addressing -> 2404-00781
  lyle2024disentangling -> 2402-18762
  kumar2023maintaining -> 2308-11958
  ... and 27 more pairs

=== 2412-19279 ===
Number of labeled pairs: 54
  lin2024preserving -> 2402-17229
  wang2023can -> 2309-06014
  zhang2023you -> 2308-03300
  barrington2023single -> 2307-07683
  wang2023cross -> 2305-13700
  ... and 49 more pairs

=== 2412-19837 ===
Number of labeled pairs: 24
  Mao2024PrivShape -> 2404-03873
  li2023 -> 2311-16062
  zhang2023 -> 2307-09339
  du2023 -> 2302-06180
  Stateful2023Optimized -> 2212-08792
  ... and 19 more pairs

=== 2412-20155 ===
Number of labeled pairs: 20
  UnitSpeech -> 2306-16083
  Mega-TTS -> 2306-03509
  LibriTTS-R -> 2305-18802
  NaturalSpeech2 -> 2304-09116
  Vall-E -> 2301-02111
  ... and 15 more pairs

=== 2412-20242 ===
Number of labeled pairs: 30
  Arroyo-Urena:2024soo -> 2405.06036
  

Kết quả thống kê cho thấy tổng cộng *160 cặp nhãn* đã được thu thập từ 5 bài báo, với trung bình *32 cặp/bài*. Số lượng này vượt qua ngưỡng tối thiểu yêu cầu (20 cặp), đảm bảo độ tin cậy thống kê cho tập *Validation* và *Test*.

### 1.2. Phân tích cấu trúc dữ liệu chi tiết

Để hiểu rõ đặc trưng của bài toán Xếp hạng (Ranking), chúng ta cần xem xét cấu trúc nguyên thô của hai nguồn dữ liệu đầu vào:
1.  **BibTeX Entries (Query):** Thông tin trích dẫn từ bài báo gốc.
2.  **ArXiv Metadata (Candidates):** Thông tin bài báo tham chiếu từ cơ sở dữ liệu.

Việc kiểm tra này giúp xác nhận `DataLoader` hoạt động chính xác và các trường thông tin quan trọng (Title, Author, Year) đã được trích xuất đầy đủ.

In [3]:
# Initialize DataLoader và load một publication mẫu
data_loader = DataLoader(DATA_DIR, PROCESSED_DIR)

# Load một manual publication mẫu
sample_manual_pub = label_config['manual_subset']['papers'][0]
print(f"=== Loading Manual Publication: {sample_manual_pub} ===")
bib_entries_manual, arxiv_entries_manual = data_loader.load_publication(sample_manual_pub, use_processed=True)

print(f"Number of BibTeX entries: {len(bib_entries_manual)}")
print(f"Number of arXiv entries: {len(arxiv_entries_manual)}")
print(f"Total possible pairs: {len(bib_entries_manual) * len(arxiv_entries_manual)}")

# Show sample entries
print(f"\n=== Sample BibTeX Entries ===")
for i, bib in enumerate(bib_entries_manual[:3]):
    print(f"\n[{i+1}] Key: {bib.key}")
    title_display = bib.title[:80] + "..." if len(bib.title) > 80 else bib.title
    print(f"    Title: {title_display}")
    authors_display = ', '.join(bib.authors[:3]) + ('...' if len(bib.authors) > 3 else '')
    print(f"    Authors: {authors_display}")
    print(f"    Year: {bib.year}")

2026-01-18 15:48:44,462 - INFO - Loaded 65 BibTeX entries from refs.bib
2026-01-18 15:48:44,473 - INFO - Loaded 31 arXiv entries from references.json


=== Loading Manual Publication: 2412-18790 ===
Number of BibTeX entries: 65
Number of arXiv entries: 31
Total possible pairs: 2015

=== Sample BibTeX Entries ===

[1] Key: Bengio+chapter2007
    Title: Scaling Learning Algorithms Towards {AI}
    Authors: Bengio, Yoshua, LeCun, Yann
    Year: 2007

[2] Key: Dohare2021Continual
    Title: Continual backprop: Stochastic gradient descent with persistent randomness
    Authors: Dohare, Shibhansh, Richard S. Sutton, and A. Rupam Mahmood
    Year: 2021

[3] Key: Hinton06
    Title: A Fast Learning Algorithm for Deep Belief Nets
    Authors: Hinton, Geoffrey E., Osindero, Simon, Teh, Yee Whye
    Year: 2006


### 1.3. Kiểm tra khả năng mở rộng và chất lượng dữ liệu

Sau khi xác thực quy trình trên tập dữ liệu thủ công, chúng ta mở rộng phạm vi kiểm tra sang tập dữ liệu *Auto Subset*.

Mục tiêu của bước này là:
1.  Đảm bảo `DataLoader` xử lý ổn định trên các bài báo ngẫu nhiên trong tập 500 bài.
2.  Phát hiện sớm các bất thường về dữ liệu, ví dụ: các bài báo thiếu file `.bib` nguồn hoặc lỗi định dạng.

Đoạn mã sau sẽ thử tải ngẫu nhiên 5 bài báo từ danh sách tự động và báo cáo trạng thái dữ liệu.

In [4]:
# Load một auto publication mẫu
sample_auto_pubs = label_config['auto_subset']['papers'][:5]
print(f"=== Loading Auto Publications Sample ===")

for pub_id in sample_auto_pubs:
    try:
        bib_entries_auto, arxiv_entries_auto = data_loader.load_publication(pub_id, use_processed=False)
        print(f"\n[{pub_id}] Bib: {len(bib_entries_auto)}, arXiv: {len(arxiv_entries_auto)}")
        if bib_entries_auto and arxiv_entries_auto:
            print(f"  Sample bib key: {bib_entries_auto[0].key}")
            print(f"  Sample arxiv: {list(arxiv_entries_auto.keys())[0]}")
    except Exception as e:
        print(f"\n[{pub_id}] Error: {e}")

2026-01-18 15:48:44,486 - WARNING - No tex/ folder found in /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15272
2026-01-18 15:48:44,487 - WARNING - References file not found: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15272/references.json
2026-01-18 15:48:44,489 - WARNING - No tex/ folder found in /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15273
2026-01-18 15:48:44,492 - WARNING - References file not found: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15273/references.json
2026-01-18 15:48:44,494 - WARNING - No tex/ folder found in /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15280
2026-01-18 15:48:44,496 - WARNING - References file not found: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15280/references.json
2026-01-18 15:48:44,498 - WARNING - No tex/ folder found in /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data/2412-15285
2026-01-18 15:48:44,499 - WARNING - References file not found: /mnt/d/3rdY_HCMUS/KHDL/NMKHDL/LAB/23120195/data

=== Loading Auto Publications Sample ===

[2412-15272] Bib: 0, arXiv: 0

[2412-15273] Bib: 0, arXiv: 0

[2412-15280] Bib: 0, arXiv: 0

[2412-15285] Bib: 0, arXiv: 0

[2412-15286] Bib: 0, arXiv: 0
